# Learning Vectorial Representations of Words


We will go **step by step** through:

1. Why one-hot vectors are limited  
2. Distributed word representations  
3. Co-occurrence matrices and SVD  
4. Continuous Bag of Words (CBOW)  
5. Skip-gram  
6. Negative sampling  
7. Contrastive estimation  
8. Hierarchical softmax  
9. GloVe  
10. Evaluating word embeddings  
11. Class exercise



## Learning goals

By the end of this notebook, you should be able to:

- explain why one-hot vectors do not capture semantic similarity,
- build a co-occurrence matrix from a tiny corpus,
- use SVD to learn dense word vectors,
- explain CBOW and Skip-gram training pairs,
- implement a small Skip-gram model with negative sampling in PyTorch,
- understand the idea behind contrastive estimation and hierarchical softmax,
- describe how GloVe combines global co-occurrence statistics with learned embeddings,
- evaluate embeddings using similarity and analogy style reasoning.

In [71]:
import math
import random
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
random.seed(7)
np.random.seed(7)

## 1. Why not one-hot vectors?

Suppose our vocabulary is:

```text
["cat", "dog", "truck", "apple"]
```

A one-hot vector for `cat` might be:

```text
[1, 0, 0, 0]
```

and for `dog`:

```text
[0, 1, 0, 0]
```

### Problem
Even though **cat** and **dog** are semantically related, one-hot vectors treat every pair of different words almost the same:
- Euclidean distance between any two different one-hot vectors is the same
- cosine similarity between any two different one-hot vectors is 0

So one-hot vectors identify words, but do **not** encode meaning.

In [72]:
vocab_demo = ["cat", "dog", "truck", "apple"]
one_hot = {w: np.eye(len(vocab_demo), dtype=int)[i] for i, w in enumerate(vocab_demo)}

def cosine(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

for pair in [("cat", "dog"), ("cat", "truck"), ("dog", "apple")]:
    a, b = one_hot[pair[0]], one_hot[pair[1]]
    print(pair, "euclidean =", np.linalg.norm(a-b), "cosine =", cosine(a, b))

('cat', 'dog') euclidean = 1.4142135623730951 cosine = 0.0
('cat', 'truck') euclidean = 1.4142135623730951 cosine = 0.0
('dog', 'apple') euclidean = 1.4142135623730951 cosine = 0.0


## 2. A tiny corpus for all later examples

We will use a very small corpus so that every step is visible.

The corpus is intentionally tiny because the goal is understanding, not state-of-the-art performance.

In [73]:
sentences = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat chased the mouse",
    "the dog chased the cat",
    "the mouse ate the cheese",
    "the dog ate the bone",
    "the cat ate the fish",
]

tokenized = [s.split() for s in sentences]
tokens = [tok for sent in tokenized for tok in sent]
vocab = sorted(set(tokens))
word_to_ix = {w: i for i, w in enumerate(vocab)}
ix_to_word = {i: w for w, i in word_to_ix.items()}

print("Vocabulary:", vocab)
print("Vocab size:", len(vocab))

Vocabulary: ['ate', 'bone', 'cat', 'chased', 'cheese', 'dog', 'fish', 'mat', 'mouse', 'on', 'rug', 'sat', 'the']
Vocab size: 13


## 3. Distributed representations

The idea is simple:

> represent each word with a **dense vector** of small dimension, such as 10, 50, 100, or 300.

Instead of one-hot vectors like:

```text
cat = [0,0,0,0,1,0,0,...]
```

we want learned dense vectors like:

```text
cat = [ 0.42, -0.15, 0.91, ...]
dog = [ 0.39, -0.12, 0.88, ...]
```

Now similar words can have similar vectors.

## 4. Count-based view: co-occurrence matrix

A classic idea is:

> a word is known by the company it keeps.

We count how often each word appears near another word.

In [74]:
window_size = 2
cooc = np.zeros((len(vocab), len(vocab)), dtype=np.float32)

for sent in tokenized:
    for i, center in enumerate(sent):
        center_ix = word_to_ix[center]
        left = max(0, i - window_size)
        right = min(len(sent), i + window_size + 1)
        for j in range(left, right):
            if i == j:
                continue
            context = sent[j]
            context_ix = word_to_ix[context]
            cooc[center_ix, context_ix] += 1

print("Co-occurrence matrix shape:", cooc.shape)
print(cooc)

Co-occurrence matrix shape: (13, 13)
[[0. 1. 1. 0. 1. 1. 1. 0. 1. 0. 0. 0. 6.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [1. 0. 0. 2. 0. 0. 0. 0. 0. 1. 0. 1. 6.]
 [0. 0. 2. 0. 0. 1. 0. 0. 1. 0. 0. 0. 4.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [1. 0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 1. 5.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1.]
 [1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 3.]
 [0. 0. 1. 0. 0. 1. 0. 1. 0. 0. 1. 2. 2.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1.]
 [0. 0. 1. 0. 0. 1. 0. 0. 0. 2. 0. 0. 4.]
 [6. 1. 6. 4. 1. 5. 1. 1. 3. 2. 1. 4. 0.]]


In [75]:
import pandas as pd
cooc_df = pd.DataFrame(cooc, index=vocab, columns=vocab)
cooc_df

,ate,bone,cat,chased,cheese,dog,fish,mat,mouse,on,rug,sat,the
ate,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,6.0
bone,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
cat,1.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,6.0
chased,0.0,0.0,2.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,4.0
cheese,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
dog,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,5.0
fish,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
mat,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
mouse,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
on,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,2.0,2.0


### What does this matrix mean?

If the row is **cat** and the column is **the**, the value tells us how often **the** appears in the context of **cat**.

This matrix is:
- high-dimensional,
- sparse,
- tied to vocabulary size.

That motivates **dimensionality reduction**.

## 5. SVD for Learning Word Representations

A core theme of this lecture is that **count-based methods** (which are often sparse and high-dimensional) can be transformed into **dense vector representations** using matrix factorization.

### 🔹 The Mathematical Core: Singular Value Decomposition (SVD)

We apply **SVD** to the co-occurrence matrix $X$. The goal is to find a low-rank approximation such that:

$$X \approx U_k \Sigma_k V_k^T$$

**Where:**
* **$X$**: The original $n \times m$ co-occurrence matrix (Words $\times$ Contexts).
* **$U_k$**: An $n \times k$ matrix. The rows represent our **dense word vectors**.
* **$\Sigma_k$**: A $k \times k$ diagonal matrix of singular values, representing the "importance" of each new dimension.
* **$V_k^T$**: A $k \times m$ matrix representing the context vectors.



### 🔹 Dimensionality Reduction
By choosing a small $k$ (e.g., 100 or 300) instead of the full vocabulary size, we:
1.  **Compress the data:** Move from millions of dimensions to a few hundred.
2.  **Capture Latent Semantics:** Filter out "noise" and keep the underlying structure of how words relate.
3.  **Handle Sparsity:** Instead of a vector full of zeros, every word gets a meaningful, dense numerical signature.

In [76]:
from numpy.linalg import svd

U, S, VT = svd(cooc, full_matrices=False)
k = 2
emb_svd = U[:, :k] * S[:k]

svd_df = pd.DataFrame(emb_svd, index=vocab, columns=[f"dim_{i+1}" for i in range(k)])
svd_df.round(3)

,dim_1,dim_2
ate,-4.960,3.606
bone,-0.999,0.409
cat,-5.224,3.559
chased,-3.819,1.881
cheese,-0.999,0.409
dog,-4.306,2.987
fish,-0.999,0.409
mat,-0.833,0.723
mouse,-2.567,1.733
on,-2.603,0.301


In [77]:
def nearest_neighbors(word, embeddings, vocab, topk=5):
    idx = word_to_ix[word]
    v = embeddings[idx]
    sims = []
    for i, w in enumerate(vocab):
        if w == word:
            continue
        score = cosine(v, embeddings[i])
        sims.append((w, score))
    sims.sort(key=lambda x: x[1], reverse=True)
    return sims[:topk]

nearest_neighbors("cat", emb_svd, vocab)

[('mouse', 0.9999912665082884),
 ('dog', 0.9999648586351921),
 ('sat', 0.9995834231163051),
 ('ate', 0.999532704770374),
 ('mat', 0.9931845547670705)]

### Interpretation

Because the corpus is tiny, the results will not be perfect. But this already shows the basic idea:

- start with raw counts,
- compress them,
- use the compressed vectors as word embeddings.

## 6. Prediction-based models

Then we move from **count-based methods** to **prediction-based methods**.

Two central models are:

- **CBOW (Continuous Bag of Words)**: predict the center word from its context
- **Skip-gram**: predict context words from the center word

## 7. CBOW training examples

Sentence:

```text
the cat sat on the mat
```

With window size 2, for center word `sat`, the context is:

```text
["the", "cat", "on", "the"]
```

CBOW tries to predict `sat` from that context bag.

In [78]:
def generate_cbow_data(tokenized_sentences, window_size=2):
    data = []
    for sent in tokenized_sentences:
        for i in range(window_size, len(sent) - window_size):
            context = sent[i-window_size:i] + sent[i+1:i+window_size+1]
            target = sent[i]
            data.append((context, target))
    return data

cbow_data = generate_cbow_data(tokenized, window_size=2)
cbow_data[:5]

[(['the', 'cat', 'on', 'the'], 'sat'),
 (['cat', 'sat', 'the', 'mat'], 'on'),
 (['the', 'dog', 'on', 'the'], 'sat'),
 (['dog', 'sat', 'the', 'rug'], 'on'),
 (['the', 'cat', 'the', 'mouse'], 'chased')]

### Simple CBOW baseline

Below is a tiny CBOW model:

1. Look up embeddings for context words  
2. Average them  
3. Use a linear layer to predict the center word  
4. Train with cross-entropy loss

### CBOW intuition with a concrete numeric example

Consider a tiny vocabulary:

| Word | Index |
|------|------:|
| I | 0 |
| like | 1 |
| deep | 2 |
| learning | 3 |

Suppose the sentence is:

```text
I like deep learning
```

For **CBOW**, we use the surrounding words to predict the missing center word.

If the target word is `deep`, then one training example can be:

```text
context = ["I", "like", "learning"]
target  = "deep"
```

After converting words to indices:

```python
context_idxs = [[0, 1, 3]]   # shape: [batch=1, context_len=3]
target_idx   = [2]
```

Inside the model:

1. `self.emb(context_idxs)` looks up an embedding vector for each context word.  
   If `embed_dim = 3`, the output shape is `[1, 3, 3]`.

   Example embeddings:

   ```text
   I         -> [0.1, 0.2, 0.3]
   like      -> [0.0, 0.5, 0.1]
   learning  -> [0.4, 0.1, 0.6]
   ```

2. `e.mean(dim=1)` averages the context vectors:

   ```text
   h = ([0.1, 0.2, 0.3] + [0.0, 0.5, 0.1] + [0.4, 0.1, 0.6]) / 3
     = [0.167, 0.267, 0.333]
   ```

   Now `h` has shape `[1, 3]`.

3. `self.linear(h)` turns that average context vector into one score per vocabulary word.  
   Example output:

   ```text
   logits = [1.2, 0.3, 2.5, 0.7]
   ```

   These correspond to the vocabulary words `[I, like, deep, learning]`.

4. The largest score is for index `2`, which is `deep`, so the model predicts the correct target word.

So the big picture is:

```text
context words -> embeddings -> average -> linear layer -> predict center word
```

This is why CBOW is called **Continuous Bag of Words**: it uses dense vectors and treats the context as a bag, so word order is ignored.


In [79]:
class SimpleCBOW(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, context_idxs):
        # context_idxs: [batch, context_len]
        e = self.emb(context_idxs)              # [batch, context_len, embed_dim]
        h = e.mean(dim=1)                       # [batch, embed_dim]
        logits = self.linear(h)                 # [batch, vocab_size]
        return logits

def make_cbow_tensors(data):
    X, y = [], []
    for context, target in data:
        X.append([word_to_ix[w] for w in context])
        y.append(word_to_ix[target])
    return torch.tensor(X), torch.tensor(y)

X_cbow, y_cbow = make_cbow_tensors(cbow_data)

cbow_model = SimpleCBOW(len(vocab), embed_dim=10)
optimizer = torch.optim.Adam(cbow_model.parameters(), lr=0.05)

for epoch in range(200):
    optimizer.zero_grad()
    logits = cbow_model(X_cbow)
    loss = F.cross_entropy(logits, y_cbow)
    loss.backward()
    optimizer.step()

print("Final CBOW loss:", float(loss))

Final CBOW loss: 0.0006079660379327834


In [80]:
cbow_embeddings = cbow_model.emb.weight.detach().numpy()
nearest_neighbors("cat", cbow_embeddings, vocab)

[('mouse', 0.48024269269834746),
 ('dog', 0.45381374015266124),
 ('sat', 0.33161532706445535),
 ('rug', 0.266049457807248),
 ('mat', 0.2654566582580421)]

## 8. Skip-gram training examples

Skip-gram flips the direction.

Instead of:

> context → center word

it uses:

> center word → each context word

For example, in

```text
the cat sat on the mat
```

if the center word is `sat`, the model creates training pairs like:

```text
(sat, the), (sat, cat), (sat, on), (sat, the)
```

### Skip-gram intuition with a matching example

Now use the same tiny sentence:

```text
I like deep learning
```

For **Skip-gram**, we reverse the direction. Instead of predicting the center word from the context, we use the center word to predict each surrounding context word.

If the center word is `deep`, then the nearby context words are:

```text
["I", "like", "learning"]
```

So Skip-gram creates separate training pairs:

```text
(deep, I)
(deep, like)
(deep, learning)
```

After converting to indices using the same vocabulary:

```text
deep      -> 2
I         -> 0
like      -> 1
learning  -> 3
```

the training pairs become:

```python
[(2, 0), (2, 1), (2, 3)]
```

Conceptually, the model works like this:

1. Look up the embedding for the center word `deep`.
2. Use that embedding to score all possible vocabulary words.
3. Train the model so that the true context words (`I`, `like`, `learning`) receive high scores.

So the learning direction is:

```text
center word -> embedding -> predict one context word
```

Notice the difference from CBOW:

- **CBOW**: many context words together predict one center word
- **Skip-gram**: one center word predicts many context words

This is why Skip-gram usually creates **more training pairs** than CBOW from the same sentence.


In [81]:
def generate_skipgram_pairs(tokenized_sentences, window_size=2):
    pairs = []
    for sent in tokenized_sentences:
        for i, center in enumerate(sent):
            left = max(0, i - window_size)
            right = min(len(sent), i + window_size + 1)
            for j in range(left, right):
                if i == j:
                    continue
                pairs.append((center, sent[j]))
    return pairs

sg_pairs = generate_skipgram_pairs(tokenized, window_size=2)
sg_pairs[:12], len(sg_pairs)

([('the', 'cat'),
  ('the', 'sat'),
  ('cat', 'the'),
  ('cat', 'sat'),
  ('cat', 'on'),
  ('sat', 'the'),
  ('sat', 'cat'),
  ('sat', 'on'),
  ('sat', 'the'),
  ('on', 'cat'),
  ('on', 'sat'),
  ('on', 'the')],
 106)

## 9. Why softmax is expensive

If the vocabulary size is \(|V|\), then predicting a full probability distribution over all words requires computing a score for every vocabulary item.

That is expensive when \(|V|\) is large.

## Three common solutions:
- negative sampling,
- contrastive estimation,
- hierarchical softmax.

## 10. Skip-gram with negative sampling

### Idea

For every correct pair \((w,c)\), create a few incorrect pairs \((w,r)\) where `r` is a randomly sampled word.

The model should:
- give a **high score** to real pairs,
- give a **low score** to fake pairs.

This replaces a large softmax with a much cheaper binary classification style objective.

In [82]:
pair_counts = Counter([c for _, c in sg_pairs])
unigram = np.array([pair_counts[w] for w in vocab], dtype=np.float64)
unigram = unigram ** 0.75
unigram = unigram / unigram.sum()

def sample_negative(num_samples):
    return np.random.choice(len(vocab), size=num_samples, p=unigram)

print("Modified unigram distribution:")
for w, p in zip(vocab, unigram.round(3)):
    print(f"{w:>6}: {p}")

Modified unigram distribution:
   ate: 0.111
  bone: 0.029
   cat: 0.104
chased: 0.082
cheese: 0.029
   dog: 0.089
  fish: 0.029
   mat: 0.029
 mouse: 0.058
    on: 0.082
   rug: 0.029
   sat: 0.082
   the: 0.248


In [83]:
class SkipGramNegSampling(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, embed_dim)
        self.out_embed = nn.Embedding(vocab_size, embed_dim)

    def forward(self, center_words, pos_context_words, neg_context_words):
        v = self.in_embed(center_words)                    # [B, D]
        u_pos = self.out_embed(pos_context_words)          # [B, D]
        u_neg = self.out_embed(neg_context_words)          # [B, K, D]

        pos_score = torch.sum(v * u_pos, dim=1)            # [B]
        pos_loss = F.logsigmoid(pos_score)

        neg_score = torch.sum(u_neg * v.unsqueeze(1), dim=2)   # [B, K]
        neg_loss = F.logsigmoid(-neg_score).sum(dim=1)

        return -(pos_loss + neg_loss).mean()

In [84]:
sg_train = [(word_to_ix[w], word_to_ix[c]) for w, c in sg_pairs]

def make_batch(data, batch_size=16, neg_k=4):
    batch = random.sample(data, batch_size)
    centers = torch.tensor([x[0] for x in batch], dtype=torch.long)
    pos = torch.tensor([x[1] for x in batch], dtype=torch.long)
    neg = torch.tensor(np.array([sample_negative(neg_k) for _ in batch]), dtype=torch.long)
    return centers, pos, neg

sg_model = SkipGramNegSampling(len(vocab), embed_dim=12)
optimizer = torch.optim.Adam(sg_model.parameters(), lr=0.03)

for epoch in range(600):
    centers, pos, neg = make_batch(sg_train, batch_size=min(16, len(sg_train)), neg_k=4)
    optimizer.zero_grad()
    loss = sg_model(centers, pos, neg)
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print(f"epoch={epoch:3d} loss={float(loss):.4f}")

epoch=  0 loss=8.1081
epoch=100 loss=2.2698
epoch=200 loss=2.2483
epoch=300 loss=2.2823
epoch=400 loss=1.7111
epoch=500 loss=1.9758


In [85]:
sg_embeddings = sg_model.in_embed.weight.detach().numpy()

for word in ["cat", "dog", "mouse", "ate"]:
    print("\nNearest to", word)
    print(nearest_neighbors(word, sg_embeddings, vocab, topk=4))


Nearest to cat
[('mouse', 0.8534598830372655), ('mat', 0.5836783118768317), ('bone', 0.5399285872466731), ('dog', 0.47622656798394997)]

Nearest to dog
[('sat', 0.6420080123939804), ('bone', 0.5181513162761532), ('cat', 0.47622656798394997), ('fish', 0.4239325825247965)]

Nearest to mouse
[('cat', 0.8534598830372655), ('bone', 0.5183531675672639), ('cheese', 0.5102142294769931), ('chased', 0.47884510932798896)]

Nearest to ate
[('chased', 0.6420813103689946), ('sat', 0.5349455573922101), ('bone', 0.4650243663661187), ('rug', 0.4459190704582424)]


### 🔹 What is happening mathematically? (Skip-Gram with Negative Sampling)

In the Negative Sampling objective, we treat the problem as a **Binary Classification** task. For every target word $w$, we look at a "real" context word $c$ and several "fake" (randomly sampled) words $r$.

#### 1. For a Real Pair $(w, c)$:
We want the probability that this pair came from the actual corpus ($z=1$) to be **high**.
Mathematically, we maximize:

$$P(z=1 \mid w,c) = \sigma(u_c^T v_w)$$

#### 2. For a Negative Pair $(w, r)$:
We want the probability that this pair is "fake" ($z=0$) to be **high**.
Using the properties of the sigmoid function, this is equivalent to maximizing:

$$P(z=0 \mid w,r) = \sigma(-u_r^T v_w)$$

---

### 🔹 The Training Intuition

The training objective acts like a "Mathematical Tug-of-War":

* **Real Pairs Together:** It increases the dot product $u_c^T v_w$, pushing the vectors of words that actually appear together to be more similar (aligned).
* **Fake Pairs Apart:** It decreases the dot product $u_r^T v_w$, pushing the target word vector away from the vectors of random words that don't belong in that context.



### 🔹 Summary
Instead of calculating a massive Softmax over the entire vocabulary, we only update the weights for:
1. The **Target word** $v_w$
2. The **Positive context word** $u_c$
3. A small number of **Negative samples** $u_r$

This makes the algorithm extremely efficient and scalable to massive datasets.

## 11. A tiny manual example of negative sampling

Suppose the real pair is:

```text
(sat, on)
```

and sampled negatives are:

```text
(sat, cheese), (sat, bone)
```

Then training says:

- increase score of `(sat, on)`
- decrease scores of `(sat, cheese)` and `(sat, bone)`

That is the whole intuition.

## 12. Contrastive estimation



### Core idea
Instead of normalizing over all vocabulary words, compare:
- a correct example,
- one or more corrupted examples.

Example:
- Positive: `He sat on a chair`
- Negative: `He sat abracadabra a chair`

We want the positive sentence (or pair) to score higher than the corrupted one by a margin.

In [86]:
def score_pair(center_word, context_word, embeddings):
    c = embeddings[word_to_ix[center_word]]
    t = embeddings[word_to_ix[context_word]]
    return float(np.dot(c, t))

pos_score = score_pair("sat", "on", sg_embeddings)
neg_score = score_pair("sat", "cheese", sg_embeddings)

margin = 1.0
hinge_loss = max(0.0, margin - (pos_score - neg_score))

print("Positive score:", round(pos_score, 4))
print("Negative score:", round(neg_score, 4))
print("Margin loss:", round(hinge_loss, 4))

Positive score: 1.0662
Negative score: 1.4271
Margin loss: 1.3609


### Interpretation

If the positive score is not sufficiently larger than the negative score, we pay loss.

This is the same general spirit as many ranking and contrastive learning objectives used today.

# 13. Class exercise

Work on the following tasks in class. Try to complete as much as possible without looking ahead.

## Exercise A: Build your own co-occurrence matrix
Use the corpus below and window size = 1.

```text
"students write code"
"students debug code"
"teachers review code"
```

Tasks:
1. Build the vocabulary.
2. Construct the co-occurrence matrix.
3. Which words seem related?

## Exercise B: Compare one-hot and SVD
1. Construct one-hot vectors for the vocabulary.
2. Compute cosine similarity between:
   - students and teachers
   - code and debug
3. Apply SVD with k = 2 and compare the new similarities.

## Exercise C: Create CBOW pairs
For the sentence

```text
"deep learning learns useful features"
```

with window size = 1:
1. list all context-target pairs for CBOW,
2. list all center-context pairs for Skip-gram.


## Exercise D: Short theory questions
1. Why is one-hot representation not enough for semantic similarity?
2. Why is softmax expensive for large vocabularies?
3. What is the main difference between CBOW and Skip-gram?

In [87]:
# Starter cell for Exercise A
exercise_sentences = [
    "students write code",
    "students debug code",
    "teachers review code",
]

exercise_tokens = [s.split() for s in exercise_sentences]
exercise_vocab = sorted(set(tok for sent in exercise_tokens for tok in sent))
exercise_word_to_ix = {w:i for i, w in enumerate(exercise_vocab)}

exercise_vocab

['code', 'debug', 'review', 'students', 'teachers', 'write']

In [88]:
# Your code here:
# Build the co-occurrence matrix with window size = 1
exercise_cooc = np.zeros((len(exercise_vocab), len(exercise_vocab)))
for sent in exercise_tokens:
    for i, center in enumerate(sent):
        center_ix = exercise_word_to_ix[center]
        left = max(0, i - 1)
        right = min(len(sent), i + 2)
        for j in range(left, right):
            if i == j:
                continue
            context = sent[j]
            context_ix = exercise_word_to_ix[context]
            exercise_cooc[center_ix, context_ix] += 1
exercise_cooc_df = pd.DataFrame(exercise_cooc, index=exercise_vocab, columns=exercise_vocab)
exercise_cooc_df

,code,debug,review,students,teachers,write
code,0.0,1.0,1.0,0.0,0.0,1.0
debug,1.0,0.0,0.0,1.0,0.0,0.0
review,1.0,0.0,0.0,0.0,1.0,0.0
students,0.0,1.0,0.0,0.0,0.0,1.0
teachers,0.0,0.0,1.0,0.0,0.0,0.0
write,1.0,0.0,0.0,1.0,0.0,0.0


### Q: Which words seem related?
A: Code is related to debug, review, and write. Teachers are related to review. Students are related to debug and write.

In [89]:
# Your code here:
# 1. Construct one-hot vectors for the vocabulary.
exercise_one_hot = {w: np.eye(len(exercise_vocab), dtype=int)[i] for i, w in enumerate(exercise_vocab)}
exercise_one_hot
# 2. Compute cosine similarity between:
#    - students and teachers
#    - code and debug
sim_students_teachers = cosine(exercise_one_hot["students"], exercise_one_hot["teachers"])
sim_code_debug = cosine(exercise_one_hot["code"], exercise_one_hot["debug"])
print("Cosine similarity between 'students' and 'teachers':", sim_students_teachers)
print("Cosine similarity between 'code' and 'debug':", sim_code_debug)
# 3. Apply SVD with k = 2 and compare the new similarities.
exercise_svd_U, exercise_svd_S, exercise_svd_VT = svd(exercise_cooc, full_matrices=False)
exercise_k = 2
exercise_emb_svd = exercise_svd_U[:, :exercise_k] * exercise_svd_S[:exercise_k]
exercise_svd_df = pd.DataFrame(exercise_emb_svd, index=exercise_vocab, columns=[f"dim_{i+1}" for i in range(exercise_k)])
exercise_svd_df.round(3)




Cosine similarity between 'students' and 'teachers': 0.0
Cosine similarity between 'code' and 'debug': 0.0


,dim_1,dim_2
code,1.716,0.000
debug,-0.000,-1.366
review,0.000,-1.000
students,1.256,-0.000
teachers,0.460,-0.000
write,-0.000,-1.366


In [90]:
# Your answer here for Exercise C:
# Write down CBOW and Skip-gram training pairs for:
# "deep learning learns useful features"
# CBOW pairs: (context, target)
cbow_pairs = [
    (["deep", "learning"], "learns"),
    (["learning", "learns"], "useful"),
    (["learns", "useful"], "features"),
    (["deep", "learns"], "learning"),
    (["learning", "useful"], "learns"),
    (["learns", "features"], "useful"),
    (["deep", "useful"], "learning"),
    (["learning", "features"], "useful"),
    (["deep", "features"], "learning"),
]
# Skip-gram pairs: (center, context)
skipgram_pairs = [
    ("deep", "learning"),
    ("deep", "learns"),
    ("learning", "deep"),
    ("learning", "useful"),
    ("learns", "deep"),
    ("learns", "useful"),
    ("useful", "learning"),
    ("useful", "features"),
    ("features", "learns"),
]

print("CBOW pairs:")
for context, target in cbow_pairs:
    print(f"Context: {context} -> Target: {target}")
print("\nSkip-gram pairs:")
for center, context in skipgram_pairs:
    print(f"Center: {center} -> Context: {context}")

CBOW pairs:
Context: ['deep', 'learning'] -> Target: learns
Context: ['learning', 'learns'] -> Target: useful
Context: ['learns', 'useful'] -> Target: features
Context: ['deep', 'learns'] -> Target: learning
Context: ['learning', 'useful'] -> Target: learns
Context: ['learns', 'features'] -> Target: useful
Context: ['deep', 'useful'] -> Target: learning
Context: ['learning', 'features'] -> Target: useful
Context: ['deep', 'features'] -> Target: learning

Skip-gram pairs:
Center: deep -> Context: learning
Center: deep -> Context: learns
Center: learning -> Context: deep
Center: learning -> Context: useful
Center: learns -> Context: deep
Center: learns -> Context: useful
Center: useful -> Context: learning
Center: useful -> Context: features
Center: features -> Context: learns


### Exercise D Answers: Short theory questions

_Q: Why is one-hot representation not enough for semantic similarity?_

A: One-hot vectors treat every pair of different words as equally dissimilar, so they do not capture semantic relationships between words.

_Q: Why is softmax expensive for large vocabularies?_

A: Softmax requires computing a score for every word in the vocabulary, which is computationally expensive when the vocabulary is large.

_Q: What is the main difference between CBOW and Skip-gram?_

A: CBOW predicts the center word from its context, while Skip-gram predicts context words from the center word.


# 20. Final takeaway

Lecture 10 builds a clean progression:

1. **One-hot vectors** are easy but do not capture meaning.  
2. **Co-occurrence matrices** capture context information.  
3. **SVD** compresses count-based information into dense vectors.  
4. **CBOW** and **Skip-gram** learn embeddings by prediction.  
5. **Negative sampling**, **contrastive estimation**, and **hierarchical softmax** make learning practical.  
